# 03 · Preprocess LiTS + figures (Kaggle runner)

Reorients every scan to RAS, resamples to 0.77 × 0.77 × 1.0 mm, crops to the body and
saves compressed NIfTI. Intensity statistics are computed on **training cases only**.
Also renders the README figures. All logic lives in the
[`livertumor`](https://github.com/yanishalloum/liver_tumor_classification) package.

**Settings:** attach both `andrewmvd/liver-tumor-segmentation` datasets, **Internet on**,
accelerator **None** (CPU is enough). Use **Save Version → Save & Run All** so it runs in
the background (~30–60 min). Then, from the version's *Output* tab, click
**New Dataset** to save `lits_preprocessed/` as a private, versioned Kaggle dataset.

In [ ]:
import shutil
import subprocess

REPO = "/kaggle/working/repo"
URL = "https://github.com/yanishalloum/liver_tumor_classification.git"


def run(*cmd):
    """Run a shell command and stop the notebook if it fails.

    A plain `!command` that fails does NOT stop the notebook: the next cells run and
    fail later with a misleading error. check=True raises immediately instead.
    """
    subprocess.run(cmd, check=True)


shutil.rmtree(REPO, ignore_errors=True)  # an old clone would make `git clone` fail
try:
    run("git", "clone", "-q", "--depth", "1", URL, REPO)
except subprocess.CalledProcessError as error:
    raise RuntimeError("git clone failed: is Internet turned on (Settings > Internet)?") from error
run("pip", "install", "-q", f"{REPO}[train,viz]")
run("git", "-C", REPO, "log", "--oneline", "-1")  # exact code version of this run
assert shutil.which("livertumor"), "the livertumor command was not installed"

In [ ]:
run(
    "livertumor",
    "preprocess",
    "--data-root",
    "/kaggle/input",
    "--split",
    f"{REPO}/splits/lits_split.json",
    "--out",
    "/kaggle/working/lits_preprocessed",
    "--workers",
    "1",
)

**Sanity check:** resampling must not change organ volumes (beyond interpolation noise).

In [ ]:
import pandas as pd

m = pd.read_csv("/kaggle/working/lits_preprocessed/manifest.csv")
m["liver_change_%"] = 100 * (m.liver_ml_after / m.liver_ml_before - 1)
print(m["liver_change_%"].describe().round(2))
print("total size:", end=" ")
!du -sh /kaggle/working/lits_preprocessed
m.head()

In [ ]:
import json
from pathlib import Path

from livertumor import viz
from livertumor.data import find_cases

out = Path("/kaggle/working/figures")
cases = {c.case_id: c for c in find_cases(["/kaggle/input"])}
stats = json.loads(Path("/kaggle/working/lits_preprocessed/preprocessing.json").read_text())
vol = Path("/kaggle/working/lits_preprocessed/volumes")

# training cases only: we never look at test patients
viz.gallery_figure([viz.load_volume(cases[i]) for i in (38, 3, 102, 33)], out)
viz.windowing_figure(viz.load_volume(cases[102]), out)
viz.orientation_figure([cases[i] for i in (0, 28, 53)], out)
viz.preprocessing_figure(
    cases[1],
    vol / "case_001_image.nii.gz",
    vol / "case_001_label.nii.gz",
    stats["intensity"],
    out,
)
print(json.dumps(stats["intensity"], indent=2))

In [ ]:
from IPython.display import Image, display

for name in ("scan_gallery", "windowing", "orientation", "preprocessing"):
    display(Image(f"/kaggle/working/figures/{name}.png"))

In [ ]:
# keep the output tidy: the dataset is lits_preprocessed/, figures are separate
shutil.rmtree(REPO)
for name in ("preprocessing.json", "manifest.csv"):
    shutil.copy(f"/kaggle/working/lits_preprocessed/{name}", "/kaggle/working/figures/")